In [ ]:
import ast
from decimal import Decimal, ROUND_DOWN

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

sns.set_style('whitegrid')

csv_paths = {
    'SARIMAX': Path('../output/optimization/sarimax_trials.csv'),
    'LightGBM': Path('../output/optimization/lightgbm_trials.csv'),
    'LSTM': Path('../output/optimization/lstm_trials.csv'),
    'Chronos-2': Path('../output/optimization/chronos2_trials.csv'),
}

# Columna de RMSE por modelo
rmse_col_map = {
    'SARIMAX': 'user_attrs_val_rmse',
    'LightGBM': 'val_rmse',
    'LSTM': 'val_rmse',
    'Chronos-2': 'rmse_val',
}

# Hiperparámetro a graficar (se prueba con y sin el prefijo 'params_' de Optuna)
hyperparam = {
    'SARIMAX': 'params_p',
    'LightGBM': 'learning_rate',
    'LSTM': 'hidden_size',
    'Chronos-2': 'trial_number',
}

models_order = list(csv_paths)
palette = dict(zip(models_order, ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']))


def first_col(df, candidates):
    """Devuelve la primera columna de `candidates` que exista en df."""
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f'Ninguna de {candidates} está en las columnas: {list(df.columns)}')


def fmt4(x):
    """Cuatro decimales truncados con coma decimal (convencion del informe)."""
    if x is None or pd.isna(x):
        return ''
    q = Decimal(str(float(x))).quantize(Decimal('0.0001'), rounding=ROUND_DOWN)
    return f'{q:.4f}'.replace('.', ',')


def _norm_cols(value):
    """Normaliza una lista de covariables guardada como texto a una tupla ordenada."""
    try:
        items = ast.literal_eval(value) if isinstance(value, str) else (value or [])
    except (ValueError, SyntaxError):
        items = []
    return tuple(sorted(map(str, items)))


def dedupe_chronos(df):
    """Una fila por combinación de covariables: mejor RMSE y, en empate, menor trial_number."""
    d = df.copy()
    d['_config'] = [(_norm_cols(p), _norm_cols(f)) for p, f in
                    zip(d['selected_past_cols'], d['selected_future_cols'])]
    d = (d.sort_values(['rmse_val', 'trial_number'])
           .drop_duplicates(subset='_config', keep='first')
           .drop(columns='_config')
           .sort_values(['rmse_val', 'trial_number']))
    print(f'Chronos-2: {len(df)} trials con RMSE finito -> {len(d)} configuraciones distintas')
    return d


def load_complete(name):
    """Carga el CSV, filtra trials COMPLETE y conserva solo los RMSE finitos."""
    df = pd.read_csv(csv_paths[name])
    if 'state' in df.columns:
        df = df[df['state'] == 'COMPLETE']
    df = df.copy()
    df['rmse_val'] = pd.to_numeric(df[rmse_col_map[name]], errors='coerce')
    return df[np.isfinite(df['rmse_val'])]


data = {name: load_complete(name) for name in models_order}

# ---- Top 3 por modelo ----
# En Chronos-2 el mismo subconjunto de covariables se repite en varios trials: se conserva
# únicamente el mejor trial de cada configuración (desempate por menor trial_number), de modo
# que los tres registros graficados sean tres configuraciones distintas.
top3 = {}
for name, d in data.items():
    if name == 'Chronos-2':
        d = dedupe_chronos(d)
    best = d.nsmallest(3, 'rmse_val')
    trial_col = first_col(best, ['trial_number', 'number'])
    top3[name] = best[[trial_col, 'rmse_val']].rename(columns={trial_col: 'trial'})
    cols = [trial_col, 'rmse_val']
    if name == 'Chronos-2':
        cols += ['selected_past_cols', 'selected_future_cols']
    print(f'{name} top 3:')
    print(best[cols].to_string(index=False, float_format=fmt4))
    print()

# ---- PLOT 1: Barras de los 3 mejores trials ----
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(models_order))
width = 0.25
for k in range(3):
    vals = [top3[m]['rmse_val'].iloc[k] if len(top3[m]) > k else np.nan for m in models_order]
    bars = ax.bar(x + (k - 1) * width, vals, width, label=f'Top {k + 1}',
                  color=sns.color_palette('Blues', 5)[4 - k], edgecolor='navy')
    ax.bar_label(bars, labels=[fmt4(v) for v in vals], fontsize=8, padding=2)
ax.set_xticks(x)
ax.set_xticklabels(models_order)
ax.set_title('Mejores 3 trials por modelo (RMSE de validación)', fontsize=14, fontweight='bold')
ax.set_ylabel('RMSE', fontsize=12)
ax.set_xlabel('Modelo', fontsize=12)
ax.set_ylim(bottom=20)  # el eje y empieza en 20
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()


# ---- PLOT 3: Violin plot general ----
violin_df = pd.concat(
    [d[['rmse_val']].rename(columns={'rmse_val': 'RMSE'}).assign(Modelo=name)
     for name, d in data.items()],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(data=violin_df, x='Modelo', y='RMSE', hue='Modelo',
               order=models_order, hue_order=models_order,
               palette=palette, inner='quartile', legend=False, ax=ax)
for idx, name in enumerate(models_order):
    ax.scatter(idx, top3[name]['rmse_val'].min(), color='red', s=40, zorder=5,
               label='Mejor trial' if idx == 0 else None)
ax.set_title('Distribución de RMSE de validación por modelo', fontsize=14, fontweight='bold')
ax.set_ylabel('RMSE', fontsize=12)
ax.set_xlabel('Modelo', fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

sns.set_style('whitegrid')

from decimal import Decimal, ROUND_DOWN


def fmt4(x):
    """Cuatro decimales truncados con coma decimal (convencion del informe)."""
    q = Decimal(str(float(x))).quantize(Decimal('0.0001'), rounding=ROUND_DOWN)
    return f'{q:.4f}'.replace('.', ',')


def fmt4r(x):
    """Cuatro decimales redondeados con coma (convencion para correlaciones)."""
    return f'{x:.4f}'.replace('.', ',')



csv_paths = {
    'SARIMAX': Path('../output/optimization/sarimax_trials.csv'),
    'LightGBM': Path('../output/optimization/lightgbm_trials.csv'),
    'LSTM': Path('../output/optimization/lstm_trials.csv'),
    'Chronos-2': Path('../output/optimization/chronos2_trials.csv'),
}

rmse_col_map = {
    'SARIMAX': 'user_attrs_val_rmse',
    'LightGBM': 'val_rmse',
    'LSTM': 'val_rmse',
    'Chronos-2': 'rmse_val',
}

models_order = list(csv_paths)

# ---- Definir EXACTAMENTE qué columnas analizar por modelo (solo numéricos para Pearson) ----
hyperparam_cols = {
    'SARIMAX': ['params_p', 'params_q', 'params_P', 'params_D', 'params_Q'],
    'LightGBM': ['learning_rate', 'num_leaves', 'max_depth', 'min_child_samples', 
                 'feature_fraction', 'bagging_fraction', 'lambda_l1', 'lambda_l2', 
                 'n_estimators', 'fourier_k_daily', 'fourier_k_weekly', 'fourier_k_yearly'],
    'LSTM': ['hidden_size', 'num_layers', 'dropout', 'batch_size', 'learning_rate', 'epochs'],
    'Chronos-2': ['n_windows'],
}

# ---- Cargar datos ----
def load_complete(name):
    """Carga el CSV, filtra trials COMPLETE y conserva solo los RMSE finitos."""
    df = pd.read_csv(csv_paths[name])
    if 'state' in df.columns:
        df = df[df['state'] == 'COMPLETE']
    df = df.copy()
    df['rmse_val'] = pd.to_numeric(df[rmse_col_map[name]], errors='coerce')
    return df[np.isfinite(df['rmse_val'])]


data = {name: load_complete(name) for name in models_order}

# ---- Crear heatmaps de correlación ----
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, name in zip(axes.ravel(), models_order):
    d = data[name].copy()
    cols_to_use = hyperparam_cols[name]
    
    # Verificar qué columnas existen en el dataframe
    cols_found = [col for col in cols_to_use if col in d.columns]
    cols_missing = [col for col in cols_to_use if col not in d.columns]
    
    if not cols_found:
        ax.text(0.5, 0.5, f'{name}\n❌ No se encontraron hiperparámetros especificados',
                ha='center', va='center', fontsize=11, transform=ax.transAxes)
        ax.set_title(f'{name} - Correlación con RMSE', fontsize=12, fontweight='bold')
        print(f"⚠️ {name}: Columnas faltantes: {cols_missing}")
        continue
    
    # Crear dataframe con hiperparámetros + rmse_val
    cols_to_corr = cols_found + ['rmse_val']
    d_corr = d[cols_to_corr].copy()
    
    # Convertir todo a numérico
    for col in d_corr.columns:
        d_corr[col] = pd.to_numeric(d_corr[col], errors='coerce')
    
    # Descartar filas con NaN en cualquiera de las columnas
    d_corr = d_corr.dropna()
    
    if len(d_corr) == 0:
        ax.text(0.5, 0.5, f'{name}\n❌ Sin datos válidos después de conversión numérica',
                ha='center', va='center', fontsize=11, transform=ax.transAxes)
        ax.set_title(f'{name} - Correlación con RMSE', fontsize=12, fontweight='bold')
        continue
    
    # Calcular correlación solo con rmse_val
    corr_with_rmse = d_corr.corr()['rmse_val'].drop('rmse_val').sort_values()
    
    # Crear heatmap (barra horizontal)
    colors = ['#d7191c' if x < 0 else '#2b83ba' for x in corr_with_rmse.values]
    bars = ax.barh(range(len(corr_with_rmse)), corr_with_rmse.values, color=colors, edgecolor='black', linewidth=0.7)
    
    ax.set_yticks(range(len(corr_with_rmse)))
    ax.set_yticklabels(corr_with_rmse.index, fontsize=9)
    ax.set_title(f'{name} - Correlación de Hiperparámetros vs RMSE', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Correlación de Pearson', fontsize=10)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_xlim(-1, 1)
    
    # Añadir valores en las barras
    for i, (idx, v) in enumerate(corr_with_rmse.items()):
        ax.text(v + 0.03 if v > 0 else v - 0.03, i, fmt4r(v), 
                va='center', ha='left' if v > 0 else 'right', fontsize=8, fontweight='bold')
    
    # Información adicional
    info_text = f"n_trials={len(d_corr)}\nHiperparámetros={len(cols_found)}\n({len(cols_missing)} no encontrados)"
    ax.text(0.98, 0.02, info_text, transform=ax.transAxes,
            fontsize=8, ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

plt.tight_layout()
plt.show()

# ---- Tabla resumen de correlaciones ----
print("\n" + "="*90)
print("RESUMEN: CORRELACIÓN DE HIPERPARÁMETROS NUMÉRICOS CON RMSE (Pearson)")
print("="*90 + "\n")

for name in models_order:
    d = data[name].copy()
    cols_to_use = hyperparam_cols[name]
    
    # Verificar qué columnas existen
    cols_found = [col for col in cols_to_use if col in d.columns]
    cols_missing = [col for col in cols_to_use if col not in d.columns]
    
    if not cols_found:
        print(f"\n❌ {name}:")
        print(f"   Sin hiperparámetros encontrados")
        print(f"   Esperados: {cols_to_use}")
        print(f"   Faltantes: {cols_missing}\n")
        continue
    
    # Crear dataframe con hiperparámetros + rmse_val
    cols_to_corr = cols_found + ['rmse_val']
    d_corr = d[cols_to_corr].copy()
    
    # Convertir todo a numérico
    for col in d_corr.columns:
        d_corr[col] = pd.to_numeric(d_corr[col], errors='coerce')
    
    d_corr = d_corr.dropna()
    
    if len(d_corr) == 0:
        print(f"\n❌ {name}: Sin datos válidos después de conversión numérica\n")
        continue
    
    corr_with_rmse = d_corr.corr()['rmse_val'].drop('rmse_val').sort_values(key=abs, ascending=False)
    
    print(f"\n✓ {name}:")
    print(f"   Trials válidos: {len(d_corr)} | Hiperparámetros analizados: {len(cols_found)}")
    if cols_missing:
        print(f"   ⚠️  No encontrados: {cols_missing}")
    print(f"\n   {'Hiperparámetro':<30} {'Correlación':>12} {'Efecto':<20}")
    print(f"   {'-'*30} {'-'*12} {'-'*20}")
    
    for hp, corr_val in corr_with_rmse.items():
        if corr_val > 0:
            effect = "↑ AUMENTA RMSE"
            color_ind = "🔴"
        elif corr_val < 0:
            effect = "↓ DISMINUYE RMSE"
            color_ind = "🟢"
        else:
            effect = "SIN EFECTO"
            color_ind = "⚪"
        print(f"   {hp:<30} {corr_val:+.4f}       {color_ind} {effect}")
    
    print()

print("="*90)
print("INTERPRETACIÓN:")
print("  🟢 Correlación NEGATIVA: aumentar ese parámetro tiende a MEJORAR el modelo (RMSE ↓)")
print("  🔴 Correlación POSITIVA: aumentar ese parámetro tiende a EMPEORAR el modelo (RMSE ↑)")
print("  |r| ~ 0.7-1.0: Correlación FUERTE")
print("  |r| ~ 0.3-0.7: Correlación MODERADA")
print("  |r| < 0.3:     Correlación DÉBIL")
print("="*90)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import ast

sns.set_style('whitegrid')

from decimal import Decimal, ROUND_DOWN


def fmt4(x):
    """Cuatro decimales truncados con coma decimal (convencion del informe)."""
    q = Decimal(str(float(x))).quantize(Decimal('0.0001'), rounding=ROUND_DOWN)
    return f'{q:.4f}'.replace('.', ',')


def fmt4r(x):
    """Cuatro decimales redondeados con coma (convencion para correlaciones)."""
    return f'{x:.4f}'.replace('.', ',')



csv_paths = {
    'SARIMAX': Path('../output/optimization/sarimax_trials.csv'),
    'LightGBM': Path('../output/optimization/lightgbm_trials.csv'),
    'LSTM': Path('../output/optimization/lstm_trials.csv'),
    'Chronos-2': Path('../output/optimization/chronos2_trials.csv'),
}

rmse_col_map = {
    'SARIMAX': 'user_attrs_val_rmse',
    'LightGBM': 'val_rmse',
    'LSTM': 'val_rmse',
    'Chronos-2': 'rmse_val',
}

features_col_map = {
    'SARIMAX': 'user_attrs_features',
    'LightGBM': 'future_groups_selected',
    'LSTM': 'selected_features',
    'Chronos-2': 'selected_future_cols',
}

models_order = list(csv_paths)

# ---- Definir covariables por modelo ----
all_features = {
    'SARIMAX': {
        'Ciclicidad temporal': ['hora_sin', 'hora_cos'],
        'Día de la semana': ['dia_semana_sin', 'dia_semana_cos'],
        'Día del año': ['dia_anio_sin', 'dia_anio_cos'],
        'Mes': ['mes_sin', 'mes_cos'],
        'Laboral': ['es_laboral'],
        'Temperatura': ['TEMP_F_24'],
        'Humedad': ['HUM_F_24'],
        'Target lag 24h': ['target_lag_24'],
        'Fourier Diario': ['fourier_daily_sin_1', 'fourier_daily_cos_1', 'fourier_daily_sin_2', 
                          'fourier_daily_cos_2', 'fourier_daily_sin_3', 'fourier_daily_cos_3',
                          'fourier_daily_sin_4', 'fourier_daily_cos_4', 'fourier_daily_sin_5', 
                          'fourier_daily_cos_5'],
        'Fourier Semanal': ['fourier_weekly_sin_1', 'fourier_weekly_cos_1', 'fourier_weekly_sin_2', 
                           'fourier_weekly_cos_2'],
        'Fourier Anual': ['fourier_yearly_sin_1', 'fourier_yearly_cos_1', 'fourier_yearly_sin_2', 
                         'fourier_yearly_cos_2'],
    },
    'LightGBM': {
        'Ciclicidad temporal': ['hora_sin', 'hora_cos'],
        'Día de la semana': ['dia_sem_sin', 'dia_sem_cos'],
        'Feriado': ['es_feriado'],
        'Índice temporal': ['indice_temporal'],
        'Temperatura': ['temperatura'],
        'Humedad': ['humedad'],
        'Temp lag 24h': ['temp_lag_24h'],
        'Hum lag 24h': ['hum_lag_24h'],
        'Rolling Temp 24h': ['rolling_mean_temp_24h'],
        'Rolling Target 24h': ['rolling_mean_24h', 'rolling_std_24h'],
        'Fourier Diario': ['fourier_daily_sin_1', 'fourier_daily_cos_1', 'fourier_daily_sin_2',
                          'fourier_daily_cos_2', 'fourier_daily_sin_3', 'fourier_daily_cos_3',
                          'fourier_daily_sin_4', 'fourier_daily_cos_4', 'fourier_daily_sin_5',
                          'fourier_daily_cos_5', 'fourier_daily_sin_6', 'fourier_daily_cos_6'],
        'Fourier Semanal': ['fourier_weekly_sin_1', 'fourier_weekly_cos_1', 'fourier_weekly_sin_2',
                           'fourier_weekly_cos_2', 'fourier_weekly_sin_3', 'fourier_weekly_cos_3',
                           'fourier_weekly_sin_4', 'fourier_weekly_cos_4'],
        'Fourier Anual': ['fourier_yearly_sin_1', 'fourier_yearly_cos_1', 'fourier_yearly_sin_2',
                         'fourier_yearly_cos_2'],
    },
    'LSTM': {
        'Ciclicidad temporal': ['hora_sin', 'hora_cos'],
        'Temperatura': ['temperatura'],
        'Humedad': ['humedad'],
        'Temp lag 24h': ['temp_lag_24h'],
        'Hum lag 24h': ['hum_lag_24h'],
        'Feriado': ['es_feriado'],
        'Día del año': ['dia_anio_sin', 'dia_anio_cos'],
        'Target lag 24h': ['target_lag_24'],
    },
    'Chronos-2': {
        'Laboral': ['es_laboral'],
        'Temperatura': ['TEMP_F_24'],
        'Target lag 24h': ['target_lag_24'],
    },
}

# ---- Cargar datos ----
def load_complete(name):
    """Carga el CSV, filtra trials COMPLETE y conserva solo los RMSE finitos."""
    df = pd.read_csv(csv_paths[name])
    if 'state' in df.columns:
        df = df[df['state'] == 'COMPLETE']
    df = df.copy()
    df['rmse_val'] = pd.to_numeric(df[rmse_col_map[name]], errors='coerce')
    return df[np.isfinite(df['rmse_val'])]


data = {name: load_complete(name) for name in models_order}

# ---- Funciones para parsear features ----
def parse_features_sarimax_lstm_chronos(feat_str):
    """Parsea string con formato: "['feat1', 'feat2', ...]" """
    if not isinstance(feat_str, str) or pd.isna(feat_str):
        return []
    try:
        return ast.literal_eval(feat_str)
    except:
        return []

def parse_features_lightgbm(feat_str):
    """Parsea string con formato: "use_feat1,use_feat2,..." """
    if not isinstance(feat_str, str) or pd.isna(feat_str):
        return []
    return [f.strip() for f in feat_str.split(',') if f.strip()]

# ---- Crear binary feature matrix por modelo ----
def create_feature_matrix(name, df):
    """Crea una matriz donde cada columna es un feature y cada fila indica si fue usado."""
    features_col = features_col_map[name]
    
    if features_col not in df.columns:
        print(f"⚠️ {name}: Columna '{features_col}' no encontrada")
        return None
    
    d = df.copy()
    
    # Parsear features según el modelo
    if name == 'LightGBM':
        d['features_list'] = d[features_col].apply(parse_features_lightgbm)
    else:
        d['features_list'] = d[features_col].apply(parse_features_sarimax_lstm_chronos)
    
    # Obtener todas las features únicas mencionadas
    all_feats_in_data = set()
    for feat_list in d['features_list']:
        all_feats_in_data.update(feat_list)
    
    # Crear variables binarias por feature
    feature_matrix = pd.DataFrame(index=d.index)
    feature_matrix['rmse_val'] = d['rmse_val'].values
    
    for feat in sorted(all_feats_in_data):
        feature_matrix[feat] = d['features_list'].apply(lambda x: 1 if feat in x else 0)
    
    return feature_matrix

# ---- Crear heatmaps de correlación de features ----
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for ax, name in zip(axes.ravel(), models_order):
    d = data[name].copy()
    feature_matrix = create_feature_matrix(name, d)
    
    if feature_matrix is None or feature_matrix.shape[1] <= 1:  # solo rmse_val
        ax.text(0.5, 0.5, f'{name}\n❌ No hay covariables encontradas',
                ha='center', va='center', fontsize=11, transform=ax.transAxes)
        ax.set_title(f'{name} - Correlación de Covariables vs RMSE', fontsize=12, fontweight='bold')
        continue
    
    # Calcular correlación con rmse_val
    corr_with_rmse = feature_matrix.corr()['rmse_val'].drop('rmse_val').sort_values()
    
    # Crear heatmap (barra horizontal)
    colors = ['#d7191c' if x < 0 else '#2b83ba' for x in corr_with_rmse.values]
    bars = ax.barh(range(len(corr_with_rmse)), corr_with_rmse.values, color=colors, edgecolor='black', linewidth=0.7)
    
    ax.set_yticks(range(len(corr_with_rmse)))
    ax.set_yticklabels(corr_with_rmse.index, fontsize=9)
    ax.set_title(f'{name} - Correlación de Covariables vs RMSE', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Correlación de Pearson', fontsize=10)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_xlim(-1, 1)
    
    # Añadir valores en las barras
    for i, (idx, v) in enumerate(corr_with_rmse.items()):
        ax.text(v + 0.03 if v > 0 else v - 0.03, i, fmt4r(v), 
                va='center', ha='left' if v > 0 else 'right', fontsize=8, fontweight='bold')
    
    # Información adicional
    info_text = f"n_trials={len(feature_matrix)}\nCovariables={len(corr_with_rmse)}"
    ax.text(0.98, 0.02, info_text, transform=ax.transAxes,
            fontsize=8, ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.6))

plt.tight_layout()
plt.show()

# ---- Tabla resumen de correlaciones de features ----
print("\n" + "="*90)
print("RESUMEN: CORRELACIÓN DE COVARIABLES CON RMSE (Pearson)")
print("="*90 + "\n")

for name in models_order:
    d = data[name].copy()
    feature_matrix = create_feature_matrix(name, d)
    
    if feature_matrix is None or feature_matrix.shape[1] <= 1:
        print(f"\n❌ {name}: No hay covariables encontradas\n")
        continue
    
    corr_with_rmse = feature_matrix.corr()['rmse_val'].drop('rmse_val').sort_values(key=abs, ascending=False)
    
    print(f"\n✓ {name}:")
    print(f"   Trials válidos: {len(feature_matrix)} | Covariables analizadas: {len(corr_with_rmse)}")
    print(f"\n   {'Covariable':<35} {'Correlación':>12} {'Efecto':<20}")
    print(f"   {'-'*35} {'-'*12} {'-'*20}")
    
    for feat, corr_val in corr_with_rmse.items():
        if corr_val > 0:
            effect = "↑ AUMENTA RMSE"
            color_ind = "🔴"
        elif corr_val < 0:
            effect = "↓ DISMINUYE RMSE"
            color_ind = "🟢"
        else:
            effect = "SIN EFECTO"
            color_ind = "⚪"
        print(f"   {feat:<35} {corr_val:+.4f}       {color_ind} {effect}")
    
    print()

print("="*90)
print("INTERPRETACIÓN:")
print("  🟢 Correlación NEGATIVA: incluir esa covariable tiende a MEJORAR el modelo (RMSE ↓)")
print("  🔴 Correlación POSITIVA: incluir esa covariable tiende a EMPEORAR el modelo (RMSE ↑)")
print("  |r| ~ 0.7-1.0: Correlación FUERTE")
print("  |r| ~ 0.3-0.7: Correlación MODERADA")
print("  |r| < 0.3:     Correlación DÉBIL")
print("="*90)